<a href="https://colab.research.google.com/github/GromovaN21/Bio_Docker/blob/main/Flavour1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
!pip install biopython

import pandas as pd
from Bio.KEGG import REST

VALID_CLASSES = ["Carbohydrate metabolism", "Energy metabolism", "Metabolism of other amino acids", "Xenobiotics biodegradation"]
SYNONYMS = {"pyruvate": "Pyruvic acid", "acetate": "Acetic acid", "lactate": "Lactic acid", "formate": "Formic acid", "butanoate": "Butyric acid"}

def get_organism_code(name):
    res = REST.kegg_find("genome", name.replace(" ", "+")).read().strip()
    if not res: return None
    first_line = res.split("\n")[0]
    if "\t" in first_line:
        content = first_line.split("\t")[1]
        return content.replace(";", ",").split(",")[0].strip()
    return None

def get_flavor_compounds(org_code):
    compounds = set()
    raw_modules = REST.kegg_link("module", org_code).read().strip()
    if not raw_modules: return []
    mod_ids = [line.split("\t")[1].replace("md:", "") for line in raw_modules.split("\n") if line and "\t" in line]

    for m_id in set(mod_ids):
        time.sleep(0.4) # Лимит API
        try:
            info = REST.kegg_get(f"md:{m_id}").read()
            lines = info.split("\n")
            cls = next((l for l in lines if l.startswith("CLASS")), "")
            name = next((l for l in lines if l.startswith("NAME")), "")

            is_valid = any(vc in cls for vc in VALID_CLASSES) or ("Amino acid metabolism" in cls and "biosynthesis" not in cls.lower())
            if is_valid and "=>" in name:
                prod = name.split("=>")[-1].strip().lower()
                compounds.add(SYNONYMS.get(prod, prod.capitalize()))
        except:
            continue
    return list(compounds)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 14.0 MB/s  0:00:00


In [18]:
import urllib.parse
from bs4 import BeautifulSoup
import requests


def get_flavor_profile(chemical_name):
  """Ищет вкусовой профиль вещества в FlavorDB и очищает текст."""
  encoded_name = urllib.parse.quote_plus(chemical_name)
  url = f"https://cosylab.iiitd.edu.in/flavordb/molecules?common_name={encoded_name}"

  headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
      )
  }

  try:
    response = requests.get(url, headers=headers, timeout=10)
    if response.status_code != 200:
      return f"Ошибка FlavorDB ({response.status_code})"

    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table")

    if not table or "Showing 0" in response.text:
      return "Вкус не найден в базе"

    rows = table.find_all("tr")[1:]

    flavors = set()
    for row in rows:
      cols = row.find_all("td")
      if len(cols) >= 3:
        # Очищаем строку от \n, \t и лишних пробелов
        clean_parts = [part.strip() for part in cols[2].stripped_strings]
        clean_flavor = ", ".join(clean_parts)
        if clean_flavor:
          flavors.add(clean_flavor)

    return ", ".join(sorted(flavors)) if flavors else "Вкус не найден в базе"

  except Exception:
    return "Ошибка при запросе"


In [17]:
import os
from IPython.display import clear_output, display
import ipywidgets as widgets
import pandas as pd

# Создаем элементы интерфейса
textarea = widgets.Textarea(
    placeholder="Вставьте список бактерий из сыворотки (каждый элемент с новой строки)",
    layout=widgets.Layout(width="50%", height="150px"),
)
button = widgets.Button(description="Обработать", button_style="success")
output = widgets.Output()



# === ФУНКЦИЯ БЛОКА 1 (KEGG) ===
def run_block_1(bacteria_input):
    if not bacteria_input:
        print("⚠️ Список бактерий пуст! Введите названия в текстовое поле.")
        return False

    kegg_data = []
    for bacteria in bacteria_input:
        print(f"🔄 KEGG: Обработка {bacteria}...")
        code = get_organism_code(bacteria)
        if code:
            compounds = get_flavor_compounds(code)
            for cp in compounds:
                kegg_data.append({"Бактерия": bacteria, "Вещество": cp})

    df_kegg = pd.DataFrame(kegg_data)
    df_kegg.to_csv("kegg_compounds.csv", index=False)
    print("✅ Блок 1 завершен! Данные KEGG сохранены в файл 'kegg_compounds.csv'")
    display(df_kegg)
    return True

# === ФУНКЦИЯ БЛОКА 2 (FlavorDB) ===
def run_block_2():
    try:
        df_input = pd.read_csv("kegg_compounds.csv")
        print(f"\n📂 Загружено записей из KEGG: {len(df_input)}")

        if "Вещество" in df_input.columns:
            df_input["Вещество"] = df_input["Вещество"].astype(str).str.strip()

        unique_compounds = df_input["Вещество"].unique()
        flavor_mapping = {}

        print(f"🔍 Начинаем опрос FlavorDB для {len(unique_compounds)} уникальных веществ...")
        for cp in unique_compounds:
            if cp == 'nan' or not cp:
                continue
            print(f"Ищем вкус для: {cp}...")
            flavor_mapping[cp] = get_flavor_profile(cp)

        df_input["Вкус/Аромат"] = df_input["Вещество"].map(flavor_mapping)
        df_input.to_csv("final_flavor_table.csv", index=False, encoding="utf-8-sig")

        print("\n=== ФИНАЛЬНАЯ ТАБЛИЦА ВКУСОВ СОХРАНЕНА ===")
        display(df_input)
    except FileNotFoundError:
        print("❌ Ошибка: Файл 'kegg_compounds.csv' не найден.")





# Логика при нажатии кнопки
# === ЛОГИКА НАЖАТИЯ ГЛАВНОЙ КНОПКИ ===
def on_button_click(b):
    with output:
        clear_output()
        items = [line.strip() for line in textarea.value.split("\n") if line.strip()]
        file_name = "kegg_compounds.csv"

        # Проверяем, существует ли уже файл с результатами KEGG
        if os.path.exists(file_name):
            print(f"⚠️ Файл '{file_name}' уже существует. Вы хотите обновить прошлый список?")

            btn_yes = widgets.Button(description="Да (удалить и обновить)", button_style="danger")
            btn_no = widgets.Button(description="Нет (использовать старый)", button_style="info")

            # Действие при клике на "Да"
            def on_yes_click(b_yes):
                with output:
                    clear_output()
                    os.remove(file_name)
                    print(f"🗑️ Старый файл '{file_name}' удален. Запускаем сбор заново...\n")
                    if run_block_1(items):
                        run_block_2()

            # Действие при клике на "Нет"
            def on_no_click(b_no):
                with output:
                    clear_output()
                    print(f"⏭️ Блок 1 пропущен. Используем существующий '{file_name}'...")
                    run_block_2()

            btn_yes.on_click(on_yes_click)
            btn_no.on_click(on_no_click)

            # Отображаем кнопки выбора рядом друг с другом
            display(widgets.HBox([btn_yes, btn_no]))
        else:
            # Если файла нет, сразу запускаем оба блока подряд
            if run_block_1(items):
                run_block_2()

button.on_click(on_button_click)

print("Для примера заквасок:\nLactococcus lactis\nLactococcus cremoris\nLactococcus diacetylactis")
display(textarea, button, output)

Для примера заквасок:
Lactococcus lactis
Lactococcus cremoris
Lactococcus diacetylactis


Textarea(value='', layout=Layout(height='150px', width='50%'), placeholder='Вставьте список бактерий из сыворо…

Button(button_style='success', description='Обработать', style=ButtonStyle())

Output()